# QNN From Scratch: Reproducible Walkthrough

This notebook demonstrates the package API without embedding execution outputs. The reported metrics from any run are a single seeded demonstration, not a benchmark or a claim of quantum advantage.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np

from qnn.config import ExperimentConfig
from qnn.datasets import make_classification_dataset, train_test_split
from qnn.metrics import confusion_matrix_binary
from qnn.qnn import DataReuploadingQNN
from qnn.trainer import Trainer, TrainingConfig

np.set_printoptions(precision=4, suppress=True)

## 1. Generate and inspect data

In [ ]:
experiment = ExperimentConfig(
    seed=7, num_samples=48, test_size=0.25, dataset="sine", noise=0.08,
    num_qubits=2, num_layers=2, num_features=2, observable_wire=1,
    init_scale=0.25, epochs=8, learning_rate=0.08, batch_size=0,
    grad_clip=2.0, log_every=2, output_dir="outputs/notebook_run",
)
X_all, y_all, raw_all = make_classification_dataset(
    num_samples=experiment.num_samples, kind=experiment.dataset,
    noise=experiment.noise, seed=experiment.seed,
)
data = train_test_split(
    X_all, y_all, raw_all, test_size=experiment.test_size, seed=experiment.seed
)
X_train, y_train = data.X_train, data.y_train
X_monitor, y_monitor = data.X_test, data.y_test

plt.figure(figsize=(6, 5))
plt.scatter(raw_all[:, 0], raw_all[:, 1], c=y_all, edgecolors="k", s=35)
plt.xlabel("raw x0")
plt.ylabel("raw x1")
plt.title("Synthetic sine dataset")
plt.show()

## 2. Build the model and verify the readout

In [ ]:
model = DataReuploadingQNN(
    num_qubits=experiment.num_qubits, num_layers=experiment.num_layers,
    num_features=experiment.num_features, observable_wire=experiment.observable_wire,
    seed=experiment.seed, init_scale=experiment.init_scale,
)
print("architecture:", model.architecture())
sample_state = model.forward_state(X_train[0])
sample_probability = model.predict_proba(X_train[:1])[0]
direct_probability = model.sim.probability_one(sample_state, model.observable_wire)
print("model p(class=1):", sample_probability)
print("direct Born marginal:", direct_probability)
np.testing.assert_allclose(sample_probability, direct_probability, atol=1e-12)

## 3. Inspect parameter-shift gradients

Each trainable scalar appears in one Pauli-generated rotation. The package applies the two-term shift rule to the quantum expectation and then uses the ordinary chain rule through the binary loss.

In [ ]:
grad0 = model.parameter_shift_gradient(X_train[:12], y_train[:12])
print("gradient shape:", grad0.shape)
print("gradient L2 norm:", np.linalg.norm(grad0))
plt.figure(figsize=(8, 3))
plt.bar(np.arange(grad0.size), grad0.ravel())
plt.xlabel("parameter index")
plt.ylabel("dL/dtheta")
plt.title("Initial gradient")
plt.show()

## 4. Train and plot learning curves

In [ ]:
cfg = TrainingConfig(
    seed=experiment.seed, epochs=experiment.epochs,
    learning_rate=experiment.learning_rate, batch_size=experiment.batch_size,
    grad_clip=experiment.grad_clip, log_every=experiment.log_every,
    output_dir=experiment.output_dir,
)
trainer = Trainer(model, cfg)
history = trainer.fit(X_train, y_train, X_monitor, y_monitor)

epochs = np.array([r.epoch for r in history])
train_loss = np.array([r.train_loss for r in history])
monitor_loss = np.array([r.test_loss for r in history])
train_acc = np.array([r.train_accuracy for r in history])
monitor_acc = np.array([r.test_accuracy for r in history])

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(epochs, train_loss, marker="o", label="train")
axes[0].plot(epochs, monitor_loss, marker="o", label="monitor")
axes[0].set(xlabel="epoch", ylabel="BCE", title="Loss")
axes[0].legend()
axes[1].plot(epochs, train_acc, marker="o", label="train")
axes[1].plot(epochs, monitor_acc, marker="o", label="monitor")
axes[1].set(xlabel="epoch", ylabel="accuracy", ylim=(0, 1.05), title="Accuracy")
axes[1].legend()
plt.show()

## 5. Evaluate the holdout set

In [ ]:
monitor_p = model.predict_proba(X_monitor)
cm = confusion_matrix_binary(monitor_p, y_monitor)
print("monitoring confusion matrix:", cm)
print("last logged record:", history[-1])

## 6. Save reproducibility artifacts

In [ ]:
out_dir = trainer.save_artifacts(
    X_train, y_train, X_monitor, y_monitor, raw_test=data.raw_test,
    experiment_config=experiment.to_dict(),
)
print("saved to:", out_dir)
summary = json.loads((Path(out_dir) / "summary.json").read_text())
summary

## Interpretation

A successful run only demonstrates that this small exact-statevector model can optimize the selected synthetic task for the chosen seed and hyperparameters. It does not establish quantum advantage, hardware robustness, or superiority to classical baselines.